# Cleaning ahead/behind sign jumps — iterative pipeline

`decode_distance` in `HexMazeDecodedPosition` is `sign(cos Δθ) * graph_distance`.
The **magnitude** (graph geodesic distance) is trustworthy; the **sign** flips
abruptly, producing +50 → -50 cm jumps between adjacent samples. We clean
iteratively and **print jump stats after each step** so the effect is visible:

1. **Baseline** — detect jumps and print stats.
2. **Rolling-median despike** of `decode_distance` → re-print stats. Removes
   single-sample spikes, brief decode teleports, and short oscillation bursts.
3. **Repair the orientation data** (de-glitch it) *instead of masking samples* —
   keeps every sample and fixes the reference heading that drives the sign.
4. **Recompute the sign with the deadband** on the cleaned orientation; magnitude
   from the despiked distance. The remaining jumps should be the lateral ones the
   deadband absorbs — "all other jumps are fine."

Jumps are labelled by mechanism (lateral decode / orientation glitch / decode
teleport / other) using the same thresholds as the fixes. Nothing here modifies the
database — it all runs on the fetched dataframe.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spyglass_hexmaze.hex_maze_decoding import HexMazeDecodedPosition, compute_aheadness

### Tuning knobs

In [ ]:
MAG_THRESH           = 20.0   # cm. A sign flip counts as a "jump" only if |distance| exceeds this on
                              # BOTH sides (a real sweep crosses zero smoothly, so it stays out).
LATERAL_CUT          = 0.2    # |aheadness| below this on both sides of a flip => lateral decode.
COS_MARGIN           = 0.3    # Deadband half-width for the cleaned sign. cos=0 is 90 deg; 0.3 ~ 72.5 deg.
ORI_JUMP_MAX         = 15.0   # deg/sample. Larger head-direction jumps are non-physical => orientation glitch.
DECODE_STEP_MAX      = 20.0   # cm/sample. Larger decode MAP jumps are non-physical => decode teleport.
MED_WINDOW           = 5      # samples. Rolling-median window for the despike step (odd; ~10 ms at 500 Hz).

### Helper functions

`compute_aheadness` is imported from `hex_maze_decoding`. The rest are specific to
this cleaning pipeline.

In [ ]:
def find_sign_flip_jumps(df, value_col="decode_distance", mag_thresh=MAG_THRESH,
                         require_adjacent=True):
    """Sign flips of value_col that stay large on both sides (artifact flips, not smooth
    zero-crossings). Returns (jumps_df, idx); idx = position of the sample BEFORE each
    flip (flip is between idx and idx+1). aheadness is recomputed from df's CURRENT
    orientation, so it reflects any orientation repair already applied."""
    d = df[value_col].to_numpy()
    ahead = compute_aheadness(df).to_numpy()
    t = df.index.to_numpy()
    prev, curr = d[:-1], d[1:]
    keep = (np.sign(prev) != np.sign(curr)) & (np.minimum(np.abs(prev), np.abs(curr)) > mag_thresh)
    if require_adjacent:
        keep &= np.diff(t) < 1.5 * np.median(np.diff(t))
    idx = np.where(keep)[0]
    jumps = pd.DataFrame({
        "time": t[idx],
        "dist_before": d[idx],           "dist_after": d[idx + 1],
        "aheadness_before": ahead[idx],  "aheadness_after": ahead[idx + 1],
    })
    jumps["abs_dist_diff"] = jumps["dist_before"].abs() - jumps["dist_after"].abs()
    jumps["max_abs_ahead"] = jumps[["aheadness_before", "aheadness_after"]].abs().max(axis=1)
    return jumps, idx


def jump_stats(df, value_col="decode_distance", label=""):
    """Detect jumps in value_col, classify each by mechanism, print a summary, and
    return (jumps, idx). Mechanism uses the same thresholds as the cleaning fixes."""
    jumps, idx = find_sign_flip_jumps(df, value_col=value_col)
    jumps["decode_step"]  = df["decode_step"].to_numpy()[idx + 1]
    jumps["ori_jump_deg"] = df["ori_jump_deg"].to_numpy()[idx + 1]
    jumps["spatial_cov"]  = df["spatial_cov"].to_numpy()[idx]
    jumps["mechanism"] = np.select(
        [jumps["ori_jump_deg"]  > ORI_JUMP_MAX,
         jumps["decode_step"]   > DECODE_STEP_MAX,
         jumps["max_abs_ahead"] < LATERAL_CUT],
        ["orientation glitch", "decode teleport", "lateral decode"], default="other")
    head = f"[{label}]  " if label else ""
    print(f"{head}{len(jumps):,} jumps  ({len(jumps) / len(df):.3%} of samples)")
    print(jumps["mechanism"].value_counts().to_string(), "\n")
    return jumps, idx


def flag_orientation_glitches(df, max_jump_deg=ORI_JUMP_MAX):
    """Boolean mask of single-sample non-physical head-orientation jumps (glitch, e.g.
    the 180 deg front/back flip). Flags the glitched sample AND its partner edge."""
    d = df["orientation"].diff().to_numpy()
    jump = pd.Series(np.degrees(np.abs(np.angle(np.exp(1j * d)))), index=df.index)  # [0,180]
    return (jump > max_jump_deg) | (jump.shift(-1) > max_jump_deg)


def repair_orientation(df, max_jump_deg=ORI_JUMP_MAX):
    """De-glitch head orientation: replace the samples flagged by flag_orientation_glitches
    with a wrap-safe interpolation of the surrounding GOOD samples. Keeps every sample
    (nothing masked) and returns a cleaned orientation Series."""
    ori = df["orientation"].to_numpy()
    good = ~flag_orientation_glitches(df, max_jump_deg).to_numpy()
    x = np.arange(len(ori))
    # interpolate on the unit circle so we don't get wraparound artefacts at +/-pi
    z = np.exp(1j * ori)
    zc = np.interp(x, x[good], z.real[good]) + 1j * np.interp(x, x[good], z.imag[good])
    return pd.Series(np.angle(zc), index=df.index)


def clean_ahead_behind(df, mag_col="decode_distance", cos_margin=COS_MARGIN, init=1.0):
    """Deadband sign * |mag_col|. Sign comes from compute_aheadness(df) (i.e. df's CURRENT
    orientation) and only switches once aheadness passes +/-cos_margin, holding the
    previous sign inside the band (Schmitt trigger). Pass a despiked magnitude column
    (e.g. 'decode_distance_med') for mag_col."""
    cos = compute_aheadness(df).to_numpy()
    event = np.where(cos > cos_margin, 1.0, np.where(cos < -cos_margin, -1.0, 0.0))
    ev_pos = np.where(event != 0, np.arange(len(event)), -1)
    last = np.maximum.accumulate(ev_pos)
    sign = np.where(last >= 0, event[np.where(last >= 0, last, 0)], init)
    return pd.Series(sign * np.abs(df[mag_col].to_numpy()), index=df.index, name="ahead_behind_clean")

## Load the dataframe

In [ ]:
HexMazeDecodedPosition()

In [ ]:
key = {"nwb_file_name": "IM-1478_20220719_.nwb"}

df = (HexMazeDecodedPosition & key).fetch1_dataframe()   # indexed by time

# Derived per-sample columns (computed once, from the RAW orientation, on the contiguous df)
df["aheadness"]    = compute_aheadness(df)                                    # raw cos(Δθ)
df["decode_step"]  = np.hypot(df["decode_position_x"].diff(),
                              df["decode_position_y"].diff())                 # cm/sample decode move
df["ori_jump_deg"] = np.degrees(np.abs(np.angle(np.exp(1j * df["orientation"].diff()))))

fs = 1.0 / np.median(np.diff(df.index))
print(f"{len(df):,} samples @ ~{fs:.0f} Hz  ({df.index[-1] - df.index[0]:.0f} s)")
df.head()

## 1. Baseline jump stats

In [ ]:
jumps, idx = jump_stats(df, "decode_distance", label="raw")

## 2. Rolling-median despike → re-print stats

A rolling median over `decode_distance` removes single-sample spikes, brief decode
teleports, and short oscillation bursts. It changes values only at the despiked
samples. Compare the mechanism counts to the baseline above.

In [ ]:
df["decode_distance_med"] = (
    df["decode_distance"].rolling(MED_WINDOW, center=True, min_periods=1).median()
)
jumps_med, idx_med = jump_stats(df, "decode_distance_med", label=f"median w={MED_WINDOW}")

## 3. Repair the orientation data (instead of masking)

The orientation-glitch jumps come from single-sample ~120-180 deg errors in the
tracked head direction. Rather than drop those samples, we **de-glitch the
orientation signal** (wrap-safe interpolation over the flagged samples) and keep
every sample. Everything downstream (aheadness, the deadband sign) then uses the
cleaned orientation.

In [ ]:
df["orientation_raw"] = df["orientation"]                 # keep the original
df["orientation"]     = repair_orientation(df, ORI_JUMP_MAX)
df["aheadness"]       = compute_aheadness(df)             # recompute from clean orientation

n_fixed = int((df["orientation"] != df["orientation_raw"]).sum())
print(f"repaired {n_fixed:,} orientation samples ({n_fixed / len(df):.3%})")

## 4. Recompute the sign with the deadband on clean orientation

Sign from the deadband on the **cleaned-orientation** aheadness; magnitude from the
**despiked** distance. The orientation glitches are gone (step 3), the brief decode
teleports/spikes are gone (step 2), and the deadband absorbs the lateral flips — so
the remaining jumps should be minimal ("all other jumps are fine").

In [ ]:
df["ahead_behind_clean"] = clean_ahead_behind(df, mag_col="decode_distance_med",
                                              cos_margin=COS_MARGIN)
jumps_clean, idx_clean = jump_stats(df, "ahead_behind_clean", label="cleaned")

## Inspect an example

Step through jumps: raw vs cleaned ahead/behind, aheadness with the deadband, and
raw vs repaired orientation.

In [ ]:
def plot_jump(n, jumps=jumps, idx=idx, pad=0.3):
    t0 = df.index[idx[n]]
    w = df.loc[t0 - pad : t0 + pad]
    dt = w.index - t0

    fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

    ax[0].plot(dt, w["decode_distance"], ".-", lw=0.8, label="raw decode_distance")
    ax[0].plot(dt, w["ahead_behind_clean"], ".-", lw=0.8, color="tab:green", label="cleaned")
    ax[0].axhline(0, color="k", lw=0.5); ax[0].legend(loc="upper right")
    ax[0].set_ylabel("ahead/behind\n(cm)")

    ax[1].plot(dt, w["aheadness"], ".-", lw=0.8, color="tab:purple")
    ax[1].axhline(0, color="k", lw=0.5)
    ax[1].axhline(COS_MARGIN, color="gray", ls=":"); ax[1].axhline(-COS_MARGIN, color="gray", ls=":")
    ax[1].set_ylabel("aheadness\ncos(Δθ)"); ax[1].set_ylim(-1.05, 1.05)

    ax[2].plot(dt, np.degrees(np.unwrap(w["orientation_raw"])), ".-", lw=0.8, color="lightgray", label="raw")
    ax[2].plot(dt, np.degrees(np.unwrap(w["orientation"])),     ".-", lw=0.8, color="tab:orange", label="repaired")
    ax[2].set_ylabel("orientation\n(deg)"); ax[2].set_xlabel("time from jump (s)"); ax[2].legend(loc="upper right")

    for a in ax:
        a.axvline(0, color="r", ls="--")
    row = jumps.iloc[n]
    fig.suptitle(f"jump #{n} @ t={t0:.3f}s  |  {row['mechanism']}  |  "
                 f"{row['dist_before']:+.0f} → {row['dist_after']:+.0f} cm")
    plt.tight_layout(); plt.show()


plot_jump(0)   # then plot_jump(1), plot_jump(2), ...